In [5]:
import os

import psycopg2
import pandas as pd
from psycopg2.extras import execute_values

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

from pinecone import Pinecone, ServerlessSpec
from huggingface_hub import InferenceClient

pd.set_option("display.max_colwidth", 200)

print("Imports loaded.")


Imports loaded.


In [6]:
# --- PostgreSQL connection (edit these for your machine) ---

DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "dbproject"      # your database name
DB_USER = "postgres"       # your Postgres user

# Either export DB_PASSWORD in your terminal, or hard-code here:
DB_PASSWORD = os.environ.get("DB_PASSWORD")

conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
)
print("Connected to PostgreSQL:", DB_NAME)


Connected to PostgreSQL: dbproject


In [7]:
# --- Load source tables from SQL (no CSV) ---

df_generic = pd.read_sql("SELECT * FROM generic_drugs;", conn)
df_reviews = pd.read_sql("SELECT * FROM drug_reviews;", conn)

print("generic_drugs rows:", len(df_generic))
print("drug_reviews rows:", len(df_reviews))

print("generic_drugs columns:", df_generic.columns.tolist())
print("drug_reviews columns:", df_reviews.columns.tolist())

display(df_generic.head())
display(df_reviews.head())


generic_drugs rows: 21362
drug_reviews rows: 3107
generic_drugs columns: ['genericname', 'packagemark', 'dosagetype', 'strength', 'manufacturer']
drug_reviews columns: ['urldrugname', 'rating', 'effectiveness', 'sideeffects', 'condition', 'benefitsreview', 'sideeffectsreview', 'commentsreview']


C:\Users\Abhay\AppData\Local\Temp\ipykernel_29176\1324576167.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_generic = pd.read_sql("SELECT * FROM generic_drugs;", conn)
C:\Users\Abhay\AppData\Local\Temp\ipykernel_29176\1324576167.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_reviews = pd.read_sql("SELECT * FROM drug_reviews;", conn)


,genericname,packagemark,dosagetype,strength,manufacturer
0,aceclofenac,arostablet100-mg,tablet,100 mg,globe pharmaceuticals ltd.
1,aceclofenac,asptablet100-mg,tablet,100 mg,quality pharmaceuticals ltd.
2,aceclofenac,avenactablet100-mg,tablet,100 mg,radiant pharmaceuticals ltd.
3,aceclofenac,biofenactablet100-mg,tablet,100 mg,biogen pharmaceuticals ltd.
4,aceclofenac,acntablet100-mg,tablet,100 mg,modern pharmaceuticals ltd.


,urldrugname,rating,effectiveness,sideeffects,condition,benefitsreview,sideeffectsreview,commentsreview
0,enalapril,4,highly effective,mild side effects,management of congestive heart failure,slowed the progression of left ventricular dysfunction into overt heart failure \n\nalone or with other agents in the managment of hypertension \n\nmangagement of congestive heart failur,"cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness","monitor blood pressure , weight and asses for resolution of fluid"
1,ortho-tri-cyclen,1,highly effective,severe side effects,birth prevention,"although this type of birth control has more cons than pros, it did help with my cramps. it's also effective with the prevention of pregnancy. (along with use of condoms as well)","heavy cycle, cramps, hot flashes, fatigue, long lasting cycles. it's only been 5 1/2 months, but i'm concidering changing to a different bc. this is my first time using any kind of bc, unfortunate...","i hate this birth control, i would not suggest this to anyone."
2,ponstel,10,highly effective,no side effects,menstrual cramps,"i was used to having cramps so badly that they would leave me balled up in bed for at least 2 days. the ponstel doesn't take the pain away completely, but takes the edge off so much that normal a...",heavier bleeding and clotting than normal.,"i took 2 pills at the onset of my menstrual cramps and then every 8-12 hours took 1 pill as needed for about 3-4 days until cramps were over. if cramps are bad, make sure to take every 8 hours on ..."
3,prilosec,3,marginally effective,mild side effects,acid reflux,the acid reflux went away for a few months after just a few days of being on the drug. the heartburn started again as soon as i stopped taking it. so i began treatment again. 6 months passed and i...,"constipation, dry mouth and some mild dizziness that would go away after medication was stopped for a few days.","i was given prilosec prescription at a dose of 45mg per day. medication was taken once, every morning before eating. each treatment duration was for 6 months."
4,lyrica,2,marginally effective,severe side effects,fibromyalgia,"i think that the lyrica was starting to help with the pain, but the side-effects were just too severe to continue.",i felt extremely drugged and dopey. could not drive at all while on this med. also had extreme ankle and feet swelling and couldn't even wear shoes.,see above


In [8]:
# --- Configure column names used for soft-keying ---

GENERIC_NAME_COL    = "genericname"    # clean generic name in generic_drugs
REVIEW_DRUGNAME_COL = "urldrugname"    # messy drug name in drug_reviews
REVIEW_RATING_COL   = "rating"         # numeric rating column

# Build ONE combined free-text review column inside pandas (still from SQL)
TEXT_COLS = ["benefitsreview", "sideeffectsreview", "commentsreview"]

for c in TEXT_COLS:
    if c not in df_reviews.columns:
        raise ValueError(
            f"Expected column '{c}' in drug_reviews table. "
            f"Actual columns: {df_reviews.columns.tolist()}"
        )

df_reviews["review"] = (
    df_reviews[TEXT_COLS]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

REVIEW_TEXT_COL = "review"  # new combined text column

# --- Build canonical lists for embeddings ---

generic_names = (
    df_generic[GENERIC_NAME_COL]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .unique()
)

review_drugnames = (
    df_reviews[REVIEW_DRUGNAME_COL]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .unique()
)

print("Unique generic names:", len(generic_names))
print("Unique review drug names:", len(review_drugnames))
print("Sample generic names:", list(generic_names[:10]))
print("Sample review drug names:", list(review_drugnames[:10]))


Unique generic names: 1508
Unique review drug names: 502
Sample generic names: ['aceclofenac', 'acemetacin', 'acetate formulation [hemodialysis solution]', 'acetazolamide', 'acetylcysteine', 'acidic component [hco3 hemodialysis solution]', 'acitretin', 'aclidinium bromide + formoterol fumarate', 'activated charcoal', 'acyclovir (injection)']
Sample review drug names: ['enalapril', 'ortho-tri-cyclen', 'ponstel', 'prilosec', 'lyrica', 'propecia', 'vyvanse', 'elavil', 'xanax', 'claritin']


In [9]:
conn.rollback()
print("Rolled back the previous failed transaction.")


Rolled back the previous failed transaction.


In [10]:
# Hard reset: drop any view/table named unified_drug_data

cur = conn.cursor()

# Drop VIEW if it exists
cur.execute("DROP VIEW IF EXISTS unified_drug_data CASCADE;")

# Drop TABLE if someone accidentally created a table with the same name earlier
cur.execute("DROP TABLE IF EXISTS unified_drug_data CASCADE;")

conn.commit()
cur.close()

print("Dropped any existing unified_drug_data (view or table).")


Dropped any existing unified_drug_data (view or table).


In [11]:
view_sql = """
CREATE VIEW unified_drug_data AS
SELECT
    g.genericName              AS genericname,
    s.review_drugname,
    s.similarity,
    r.rating,
    r.effectiveness,
    r.sideeffects,
    r.condition,
    r.benefitsreview,
    r.sideeffectsreview,
    r.commentsreview
FROM drug_reviews r
JOIN drugname_softkey s
    ON LOWER(r.urldrugname) = s.review_drugname
JOIN generic_drugs g
    ON LOWER(g.genericName) = s.genericname;
"""

cur = conn.cursor()
cur.execute(view_sql)
conn.commit()
cur.close()

print("Created view unified_drug_data with the new definition.")


Created view unified_drug_data with the new definition.


In [12]:
df_unified = pd.read_sql("SELECT * FROM unified_drug_data LIMIT 5;", conn)
display(df_unified)
print("Columns:", df_unified.columns.tolist())


C:\Users\Abhay\AppData\Local\Temp\ipykernel_29176\3702070004.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_unified = pd.read_sql("SELECT * FROM unified_drug_data LIMIT 5;", conn)


,genericname,review_drugname,similarity,rating,effectiveness,sideeffects,condition,benefitsreview,sideeffectsreview,commentsreview
0,enalapril maleate,enalapril,0.795280,4,highly effective,mild side effects,management of congestive heart failure,slowed the progression of left ventricular dysfunction into overt heart failure \n\nalone or with other agents in the managment of hypertension \n\nmangagement of congestive heart failur,"cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness","monitor blood pressure , weight and asses for resolution of fluid"
1,enalapril maleate,enalapril,0.795280,4,highly effective,mild side effects,management of congestive heart failure,slowed the progression of left ventricular dysfunction into overt heart failure \n\nalone or with other agents in the managment of hypertension \n\nmangagement of congestive heart failur,"cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness","monitor blood pressure , weight and asses for resolution of fluid"
2,enalapril maleate,enalapril,0.795280,4,highly effective,mild side effects,management of congestive heart failure,slowed the progression of left ventricular dysfunction into overt heart failure \n\nalone or with other agents in the managment of hypertension \n\nmangagement of congestive heart failur,"cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness","monitor blood pressure , weight and asses for resolution of fluid"
3,l-ornithine l-aspartate,ortho-tri-cyclen,0.502211,1,highly effective,severe side effects,birth prevention,"although this type of birth control has more cons than pros, it did help with my cramps. it's also effective with the prevention of pregnancy. (along with use of condoms as well)","heavy cycle, cramps, hot flashes, fatigue, long lasting cycles. it's only been 5 1/2 months, but i'm concidering changing to a different bc. this is my first time using any kind of bc, unfortunate...","i hate this birth control, i would not suggest this to anyone."
4,l-ornithine l-aspartate,ortho-tri-cyclen,0.502211,1,highly effective,severe side effects,birth prevention,"although this type of birth control has more cons than pros, it did help with my cramps. it's also effective with the prevention of pregnancy. (along with use of condoms as well)","heavy cycle, cramps, hot flashes, fatigue, long lasting cycles. it's only been 5 1/2 months, but i'm concidering changing to a different bc. this is my first time using any kind of bc, unfortunate...","i hate this birth control, i would not suggest this to anyone."


Columns: ['genericname', 'review_drugname', 'similarity', 'rating', 'effectiveness', 'sideeffects', 'condition', 'benefitsreview', 'sideeffectsreview', 'commentsreview']


In [13]:
# --- Load sentence-transformer model ---

model_name = "all-MiniLM-L6-v2"  # small, fast, good-quality model
model = SentenceTransformer(model_name)
print("Loaded model:", model_name)


Loaded model: all-MiniLM-L6-v2


In [14]:
# --- Encode generic names and review drug names ---

generic_emb = model.encode(
    list(generic_names),
    show_progress_bar=True,
    convert_to_numpy=True,
)

review_emb = model.encode(
    list(review_drugnames),
    show_progress_bar=True,
    convert_to_numpy=True,
)

print("generic_emb shape:", generic_emb.shape)
print("review_emb shape:", generic_emb.shape)


Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Batches: 100%|██████████| 16/16 [00:00<00:00, 49.01it/s]

generic_emb shape: (1508, 384)
review_emb shape: (1508, 384)


In [15]:
# --- Build soft-key mapping using cosine similarity ---

SIMILARITY_THRESHOLD = 0.50   # tune this in your report

# cosine_similarity returns a matrix [n_review_names, n_generic_names]
sim_matrix = cosine_similarity(review_emb, generic_emb)

best_generic_idx = sim_matrix.argmax(axis=1)
best_scores = sim_matrix[np.arange(sim_matrix.shape[0]), best_generic_idx]

rows = []
for review_name, g_idx, score in zip(review_drugnames, best_generic_idx, best_scores):
    score = float(score)
    if score >= SIMILARITY_THRESHOLD:
        generic_name = generic_names[g_idx]   # lowercase
        rows.append((review_name, generic_name, score))

df_softkeys = pd.DataFrame(
    rows,
    columns=["review_drugname", "genericname", "similarity"],
)

print("Soft-key rows above threshold:", len(df_softkeys))
display(df_softkeys.head(20))


Soft-key rows above threshold: 366


,review_drugname,genericname,similarity
0,enalapril,enalapril maleate,0.795280
1,ortho-tri-cyclen,l-ornithine l-aspartate,0.502211
2,prilosec,prucalopride succinate,0.635474
3,propecia,propofol,0.599755
4,vyvanse,vinorelbine tartrate,0.504644
5,elavil,elagolix,0.550765
6,xanax,diazepam,0.625819
7,claritin,clarithromycin,0.735702
8,ambien,midazolam,0.557020
9,dextroamphetamine,dextrose,0.682721


In [16]:
# --- Download unified_drug_data as CSV for report/submission ---

df_unified = pd.read_sql("SELECT * FROM unified_drug_data;", conn)
print("Rows in unified_drug_data:", len(df_unified))
display(df_unified.head())

df_unified.to_csv("unified_drug_data.csv", index=False)
print("Saved unified_drug_data.csv in this folder.")


C:\Users\Abhay\AppData\Local\Temp\ipykernel_29176\3389457051.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_unified = pd.read_sql("SELECT * FROM unified_drug_data;", conn)


Rows in unified_drug_data: 65408


,genericname,review_drugname,similarity,rating,effectiveness,sideeffects,condition,benefitsreview,sideeffectsreview,commentsreview
0,enalapril maleate,enalapril,0.795280,4,highly effective,mild side effects,management of congestive heart failure,slowed the progression of left ventricular dysfunction into overt heart failure \n\nalone or with other agents in the managment of hypertension \n\nmangagement of congestive heart failur,"cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness","monitor blood pressure , weight and asses for resolution of fluid"
1,enalapril maleate,enalapril,0.795280,4,highly effective,mild side effects,management of congestive heart failure,slowed the progression of left ventricular dysfunction into overt heart failure \n\nalone or with other agents in the managment of hypertension \n\nmangagement of congestive heart failur,"cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness","monitor blood pressure , weight and asses for resolution of fluid"
2,enalapril maleate,enalapril,0.795280,4,highly effective,mild side effects,management of congestive heart failure,slowed the progression of left ventricular dysfunction into overt heart failure \n\nalone or with other agents in the managment of hypertension \n\nmangagement of congestive heart failur,"cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness","monitor blood pressure , weight and asses for resolution of fluid"
3,l-ornithine l-aspartate,ortho-tri-cyclen,0.502211,1,highly effective,severe side effects,birth prevention,"although this type of birth control has more cons than pros, it did help with my cramps. it's also effective with the prevention of pregnancy. (along with use of condoms as well)","heavy cycle, cramps, hot flashes, fatigue, long lasting cycles. it's only been 5 1/2 months, but i'm concidering changing to a different bc. this is my first time using any kind of bc, unfortunate...","i hate this birth control, i would not suggest this to anyone."
4,l-ornithine l-aspartate,ortho-tri-cyclen,0.502211,1,highly effective,severe side effects,birth prevention,"although this type of birth control has more cons than pros, it did help with my cramps. it's also effective with the prevention of pregnancy. (along with use of condoms as well)","heavy cycle, cramps, hot flashes, fatigue, long lasting cycles. it's only been 5 1/2 months, but i'm concidering changing to a different bc. this is my first time using any kind of bc, unfortunate...","i hate this birth control, i would not suggest this to anyone."


Saved unified_drug_data.csv in this folder.


In [17]:
import torch
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())


torch version: 2.5.1+cu121
CUDA available: True
Device count: 1


In [18]:
# --- Build documents from unified_drug_data for RAG ---

df_unified = pd.read_sql("SELECT * FROM unified_drug_data;", conn)
print("Rows in unified_drug_data:", len(df_unified))
display(df_unified.head())

def row_to_doc(row):
    parts = []
    parts.append(f"genericname: {row.genericname}")
    if hasattr(row, "condition") and row.condition:
        parts.append(f"condition: {row.condition}")
    if hasattr(row, "rating") and row.rating is not None:
        parts.append(f"rating: {row.rating}")
    if hasattr(row, "effectiveness") and row.effectiveness:
        parts.append(f"effectiveness: {row.effectiveness}")
    if hasattr(row, "sideeffects") and row.sideeffects:
        parts.append(f"sideeffects: {row.sideeffects}")
    for col in ["benefitsreview", "sideeffectsreview", "commentsreview"]:
        if hasattr(row, col):
            val = getattr(row, col)
            if val:
                parts.append(f"{col}: {val}")
    return " | ".join(str(p) for p in parts if p)

docs = [row_to_doc(r) for r in df_unified.itertuples(index=False)]
print("Number of docs:", len(docs))

# --- Use GPU if available for SentenceTransformer ---

import torch

model_name = "all-MiniLM-L6-v2"

# If you already created `model` earlier for soft-keys, you can skip reloading and just move it:
# model = SentenceTransformer(model_name)   # only if not already defined

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device for RAG embeddings:", device)

model = model.to(device)

# Larger batch_size to better use GPU; tune if you get CUDA OOM
doc_embs = model.encode(
    docs,
    batch_size=128,              # was default 32; bigger = faster on GPU
    show_progress_bar=True,
    convert_to_numpy=True,
)

print("doc_embs shape:", doc_embs.shape)


C:\Users\Abhay\AppData\Local\Temp\ipykernel_29176\1852470027.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_unified = pd.read_sql("SELECT * FROM unified_drug_data;", conn)


Rows in unified_drug_data: 65408


,genericname,review_drugname,similarity,rating,effectiveness,sideeffects,condition,benefitsreview,sideeffectsreview,commentsreview
0,enalapril maleate,enalapril,0.795280,4,highly effective,mild side effects,management of congestive heart failure,slowed the progression of left ventricular dysfunction into overt heart failure \n\nalone or with other agents in the managment of hypertension \n\nmangagement of congestive heart failur,"cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness","monitor blood pressure , weight and asses for resolution of fluid"
1,enalapril maleate,enalapril,0.795280,4,highly effective,mild side effects,management of congestive heart failure,slowed the progression of left ventricular dysfunction into overt heart failure \n\nalone or with other agents in the managment of hypertension \n\nmangagement of congestive heart failur,"cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness","monitor blood pressure , weight and asses for resolution of fluid"
2,enalapril maleate,enalapril,0.795280,4,highly effective,mild side effects,management of congestive heart failure,slowed the progression of left ventricular dysfunction into overt heart failure \n\nalone or with other agents in the managment of hypertension \n\nmangagement of congestive heart failur,"cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness","monitor blood pressure , weight and asses for resolution of fluid"
3,l-ornithine l-aspartate,ortho-tri-cyclen,0.502211,1,highly effective,severe side effects,birth prevention,"although this type of birth control has more cons than pros, it did help with my cramps. it's also effective with the prevention of pregnancy. (along with use of condoms as well)","heavy cycle, cramps, hot flashes, fatigue, long lasting cycles. it's only been 5 1/2 months, but i'm concidering changing to a different bc. this is my first time using any kind of bc, unfortunate...","i hate this birth control, i would not suggest this to anyone."
4,l-ornithine l-aspartate,ortho-tri-cyclen,0.502211,1,highly effective,severe side effects,birth prevention,"although this type of birth control has more cons than pros, it did help with my cramps. it's also effective with the prevention of pregnancy. (along with use of condoms as well)","heavy cycle, cramps, hot flashes, fatigue, long lasting cycles. it's only been 5 1/2 months, but i'm concidering changing to a different bc. this is my first time using any kind of bc, unfortunate...","i hate this birth control, i would not suggest this to anyone."


Number of docs: 65408
Using device for RAG embeddings: cuda


Batches: 100%|██████████| 511/511 [03:10<00:00,  2.68it/s]


doc_embs shape: (65408, 384)


In [ ]:
# --- Create / connect to Pinecone RAG index and upload documents ---

PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY", "")
if PINECONE_API_KEY.startswith("YOUR_"):
    raise ValueError("Set PINECONE_API_KEY in your environment or replace the placeholder.")

pc = Pinecone(api_key=PINECONE_API_KEY)

rag_index_name = "dbms-rag"

existing = [idx["name"] for idx in pc.list_indexes()]
if rag_index_name not in existing:
    pc.create_index(
        name=rag_index_name,
        dimension=doc_embs.shape[1],
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print("Created Pinecone index:", rag_index_name)
else:
    print("Using existing Pinecone index:", rag_index_name)

rag_index = pc.Index(rag_index_name)

vectors = []
for i, (row, emb, text) in enumerate(zip(df_unified.itertuples(index=False), doc_embs, docs)):
    vec_id = f"row-{i}"
    metadata = {
        "genericname": str(row.genericname).lower(),
        "condition": getattr(row, "condition", "") or "",
        "rating": float(getattr(row, "rating", 0)) if getattr(row, "rating", None) is not None else None,
        "text": text,
    }
    vectors.append((vec_id, emb.tolist(), metadata))

batch_size = 200
for i in range(0, len(vectors), batch_size):
    rag_index.upsert(vectors=vectors[i:i+batch_size])

print("Upserted", len(vectors), "vectors into Pinecone rag index.")


Created Pinecone index: dbms-rag
Upserted 65408 vectors into Pinecone rag index.


In [ ]:
# --- Hugging Face LLaMA client ---

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN.startswith("YOUR_"):
    raise ValueError("Set HF_TOKEN in your environment or replace the placeholder.")

LLM_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

hf_client = InferenceClient(model=LLM_MODEL_ID, token=HF_TOKEN)

def call_llm_with_context(generic_name: str, question: str, context_rows: list[str]) -> str:
    """
    LLM is forced to answer ONLY from given context_rows (table-like text).
    If answer not present, it must say so explicitly.
    """
    table_text = "\n\n".join(context_rows) if context_rows else "(no rows)"

    system_prompt = (
        "You are a helpful assistant answering questions about a drug using ONLY the given table rows.\n"
        "Each row comes from a SQL view called unified_drug_data with columns like:\n"
        "genericname, condition, rating, benefitsreview, sideeffectsreview, commentsreview.\n"
        "You must:\n"
        "1) Carefully read these rows as if they are a small table from a database.\n"
        "2) Answer the user's question ONLY using information from these rows.\n"
        "3) If the answer cannot be found in the rows, you MUST reply exactly:\n"
        "   'I could not find an answer from the database for this question.'\n"
        "Do not use outside medical knowledge. Do not guess or invent facts."
    )

    user_prompt = (
        f"Drug: {generic_name}\n"
        f"Question: {question}\n\n"
        "Rows from unified_drug_data (treat this as a table):\n"
        f"{table_text}\n\n"
        "Now answer the question using ONLY these rows."
    )

    resp = hf_client.chat_completion(
        model=LLM_MODEL_ID,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=512,
        temperature=0.2,
    )

    try:
        content = resp.choices[0].message["content"]
        if isinstance(content, list):
            parts = []
            for c in content:
                if isinstance(c, dict) and "text" in c:
                    parts.append(c["text"])
                else:
                    parts.append(str(c))
            return "".join(parts).strip()
        return str(content).strip()
    except Exception as e:
        return f"LLM call failed: {e}"


In [ ]:
# --- Manual overrides for bad brand->generic mappings ---
# These are project-level corrections, not medical truth.
BRAND_OVERRIDES = {
    "vasotec": "enalapril maleate",   # Vasotec -> enalapril
    "elavil": "amitriptyline",        # Elavil -> amitriptyline
    # add more if you find more wrong mappings
}


In [55]:
def retrieve_context_for_question(input_generic_name: str, question: str, top_k: int = 5):
    # 1) Resolve to canonical genericname stored in DB
    canonical_name = resolve_generic_name(input_generic_name)
    if canonical_name is None:
        # DB has nothing even loosely matching this name
        empty_df = pd.DataFrame(columns=["genericname"])
        return [], empty_df, None

    # 2) Fetch rows for that canonical genericname
    sql = """
        SELECT *
        FROM unified_drug_data
        WHERE LOWER(genericname) = LOWER(%s)
        LIMIT 50;
    """
    df = pd.read_sql(sql, conn, params=[canonical_name])
    if df.empty:
        return [], df, canonical_name

    # 3) Build a query vector conditioned on canonical name + question
    query_text = f"{canonical_name}. Question: {question}"
    q_emb = model.encode([query_text], convert_to_numpy=True)[0].tolist()

    # 4) Query Pinecone, filtered by canonical genericname
    result = rag_index.query(
        vector=q_emb,
        top_k=top_k,
        include_metadata=True,
        filter={"genericname": canonical_name.lower()},
    )

    rows_text = []
    for match in result.matches:
        md = match.metadata or {}
        rows_text.append(
            f"genericname={md.get('genericname','')}, "
            f"condition={md.get('condition','')}, "
            f"rating={md.get('rating','')}, "
            f"text={md.get('text','')}"
        )

    return rows_text, df, canonical_name


In [56]:
# --- Brand overrides for obviously-known pairs (optional, but useful) ---

BRAND_OVERRIDES = {
    "vasotec": "enalapril maleate",   # Vasotec -> enalapril
    "elavil": "amitriptyline",        # Elavil -> amitriptyline
    # add more here if *you* are sure of the mapping
}

from difflib import SequenceMatcher

DEFAULT_EXPLANATION_QUESTION = (
    "Explain what people in this dataset say about how the drug is used, "
    "its effectiveness, and common side effects."
)

def name_similarity(a: str, b: str) -> float:
    a = (a or "").lower().strip()
    b = (b or "").lower().strip()
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def get_rows_for_generic(canonical_name: str, limit: int = 50) -> pd.DataFrame:
    """
    Pull up to `limit` rows from unified_drug_data for a given generic name.
    """
    sql = """
        SELECT *
        FROM unified_drug_data
        WHERE LOWER(genericname) = LOWER(%s)
        LIMIT %s;
    """
    return pd.read_sql(sql, conn, params=[canonical_name, limit])


def summarize_rows_for_drug(
    input_name: str,
    canonical_name: str,
    df: pd.DataFrame,
    question: str = DEFAULT_EXPLANATION_QUESTION,
) -> str:
    """
    Summarise what the table says about a drug with name `canonical_name`,
    given rows in df. The user originally typed `input_name`.
    """

    context_rows = []
    for row in df.itertuples(index=False):
        parts = [
            f"genericname={getattr(row, 'genericname', '')}",
            f"review_drugname={getattr(row, 'review_drugname', '')}",
            f"condition={getattr(row, 'condition', '')}",
            f"rating={getattr(row, 'rating', '')}",
            f"effectiveness={getattr(row, 'effectiveness', '')}",
            f"sideeffects={getattr(row, 'sideeffects', '')}",
            f"benefitsreview={getattr(row, 'benefitsreview', '')}",
            f"sideeffectsreview={getattr(row, 'sideeffectsreview', '')}",
            f"commentsreview={getattr(row, 'commentsreview', '')}",
        ]
        context_rows.append(" | ".join(str(p) for p in parts if p))

    table_text = "\n\n".join(context_rows)

    system_prompt = (
        "You are summarising user reviews from a database for a drug.\n"
        "Each row is from a SQL view called unified_drug_data with columns like:\n"
        "genericname, review_drugname, condition, rating, effectiveness, sideeffects,\n"
        "benefitsreview, sideeffectsreview, commentsreview.\n"
        "You must ONLY use the information in these rows. Do NOT invent medical facts.\n"
        "You are NOT giving medical advice, just summarising what people in the dataset say."
    )

    user_prompt = (
        f"The user typed the name: '{input_name}'.\n"
        f"This has been mapped in the database to the generic drug name: '{canonical_name}'.\n\n"
        f"Question: {question}\n\n"
        "Here are some rows from the database:\n"
        f"{table_text}\n\n"
        "First, clearly state the generic name you are using.\n"
        "Then, based only on these rows, summarise:\n"
        "- what conditions people use this drug for\n"
        "- what they say about its effectiveness\n"
        "- common side effects they mention\n"
        "Keep it short and clear."
    )

    resp = hf_client.chat_completion(
        model=LLM_MODEL_ID,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=512,
        temperature=0.2,
    )

    try:
        content = resp.choices[0].message["content"]
        if isinstance(content, list):
            parts = []
            for c in content:
                if isinstance(c, dict) and "text" in c:
                    parts.append(c["text"])
                else:
                    parts.append(str(c))
            return "".join(parts).strip()
        return str(content).strip()
    except Exception as e:
        return f"LLM summary failed: {e}"


def resolve_generic_name(input_name: str) -> str | None:
    """
    Try to map a common/brand name to a genericname in unified_drug_data.

    Priority:
      0) Manual override (BRAND_OVERRIDES)
      1) Co-occurrence: which generic appears most with this review_drugname?
      2) Fallback: LIKE search on genericname, then review_drugname.
      If nothing good is found, return None.
    """
    name_norm = input_name.strip().lower()

    # 0) Manual overrides
    if name_norm in BRAND_OVERRIDES:
        return BRAND_OVERRIDES[name_norm]

    # 1) Co-occurrence: which generics show up with this review_drugname?
    sql_co = """
        SELECT genericname, COUNT(*) AS n
        FROM unified_drug_data
        WHERE LOWER(review_drugname) LIKE LOWER(%s)
        GROUP BY genericname
        ORDER BY n DESC;
    """
    df_co = pd.read_sql(sql_co, conn, params=[f"%{input_name}%"])

    if not df_co.empty:
        if len(df_co) == 1:
            return df_co["genericname"].iloc[0]
        else:
            top = df_co.iloc[0]
            second_n = df_co["n"].iloc[1] if len(df_co) > 1 else 0
            # strong winner: at least 3 rows and >= 2x the second best
            if top["n"] >= 3 and top["n"] >= 2 * max(second_n, 1):
                return top["genericname"]

    # 2) Fallback: LIKE on genericname
    sql1 = """
        SELECT DISTINCT genericname
        FROM unified_drug_data
        WHERE LOWER(genericname) LIKE LOWER(%s)
        ORDER BY genericname
        LIMIT 1;
    """
    df1 = pd.read_sql(sql1, conn, params=[f"%{input_name}%"])
    if not df1.empty:
        return df1["genericname"].iloc[0]

    # 3) Fallback: LIKE on review_drugname -> genericname
    sql2 = """
        SELECT DISTINCT genericname
        FROM unified_drug_data
        WHERE LOWER(review_drugname) LIKE LOWER(%s)
        ORDER BY genericname
        LIMIT 1;
    """
    df2 = pd.read_sql(sql2, conn, params=[f"%{input_name}%"])
    if not df2.empty:
        return df2["genericname"].iloc[0]

    # 4) Nothing reliable
    return None

In [57]:
def explain_drug_from_common_name(common_name: str, max_rows: int = 40) -> str:
    """
    Main entry point for you:

    - Input: common or brand name.
    - Output: summary based on a single best genericname from the table.
    - If we cannot reliably find a generic in the table, say so.
    """
    canonical = resolve_generic_name(common_name)

    if canonical is None:
        return (
            f"I could not reliably map the name '{common_name}' to any genericname "
            f"in the unified_drug_data view, so I cannot answer this from the database."
        )

    df = get_rows_for_generic(canonical, limit=max_rows)
    if df.empty:
        return (
            f"I resolved '{common_name}' to the generic name '{canonical}' in the database, "
            f"but there are no rows for it in unified_drug_data, so I cannot answer from the database."
        )

    return summarize_rows_for_drug(common_name, canonical, df)


In [52]:
answer = explain_drug_from_common_name("glucobay")
print(answer)

I could not find any reviews in the unified_drug_data view for the name 'glucobay'. So I cannot answer this from the database.


C:\Users\Abhay\AppData\Local\Temp\ipykernel_29176\1529429691.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql(sql1, conn, params=[f"%{input_name}%"])
C:\Users\Abhay\AppData\Local\Temp\ipykernel_29176\1529429691.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql(sql2, conn, params=[f"%{input_name}%"])
C:\Users\Abhay\AppData\Local\Temp\ipykernel_29176\3957194787.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=[f"%{brand_name}%", limit])


In [42]:

answer = explain_drug_from_common_name("glucobay")
print(answer)


I could not find any matching generic drug in the unified_drug_data view for the name 'glucobay'. The internal reviews table has no data for this name.


C:\Users\Abhay\AppData\Local\Temp\ipykernel_29176\3635283156.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql(sql1, conn, params=[f"%{input_name}%"])
C:\Users\Abhay\AppData\Local\Temp\ipykernel_29176\3635283156.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql(sql2, conn, params=[f"%{input_name}%"])


In [58]:
answer = explain_drug_from_common_name("elavil")
print(answer)


I resolved 'elavil' to the generic name 'amitriptyline' in the database, but there are no rows for it in unified_drug_data, so I cannot answer from the database.


C:\Users\Abhay\AppData\Local\Temp\ipykernel_29176\452194960.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=[canonical_name, limit])


In [61]:
answer = explain_drug_from_common_name("latisse")
print(answer)


C:\Users\Abhay\AppData\Local\Temp\ipykernel_29176\452194960.py:137: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_co = pd.read_sql(sql_co, conn, params=[f"%{input_name}%"])
C:\Users\Abhay\AppData\Local\Temp\ipykernel_29176\452194960.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=[canonical_name, limit])


The generic name being used is: latanoprost.

Based on the provided rows, here is a summary:

- Conditions people use this drug for: sparse eyelashes, eyelash loss, and grow lashes.
- What they say about its effectiveness: People report that latanoprost is "considerably effective" in growing thicker, longer, and darker eyelashes, with noticeable results within 2 months.
- Common side effects they mention: Mild side effects include redness, itchiness, and occasional slight itching after application, which usually resolve within a few weeks.


In [62]:
answer = explain_drug_from_common_name("Xalatan")
print(answer)


C:\Users\Abhay\AppData\Local\Temp\ipykernel_29176\452194960.py:137: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_co = pd.read_sql(sql_co, conn, params=[f"%{input_name}%"])
C:\Users\Abhay\AppData\Local\Temp\ipykernel_29176\452194960.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=[canonical_name, limit])


The generic name being used is valsartan.

Based on the provided rows, here is a summary:

- People use valsartan for glaucoma, specifically juvenile/pigmentary (open angle) glaucoma.
- They report that valsartan is considerably effective in lowering intraocular pressure and keeping it under control.
- Common side effects mentioned include a slight change in the color of the treated eye, but these are described as mild side effects.
